In [ ]:
# MiniViGPT - train from scratch on Kaggle T4
import shutil, sys, subprocess
from pathlib import Path

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")

# Kaggle may mount inputs one dir per dataset, or nested under a single dir,
# so search recursively for the package rather than assuming a layout.
print("input tree:")
for q in sorted(INPUT.rglob("*"))[:40]:
    print("  ", q)

hits = sorted(INPUT.rglob("minivigpt/train.py"))
if not hits:
    raise SystemExit("minivigpt/train.py not found under /kaggle/input")
SRC = hits[0].parent.parent
print("source dataset:", SRC)

cfg_hits = sorted(INPUT.rglob("config.yaml"))
CONFIG_SRC = cfg_hits[0] if cfg_hits else None
print("config:", CONFIG_SRC)

# Corpus dir = wherever train.bin lives.
bin_hits = sorted(INPUT.rglob("train.bin"))
CORPUS = bin_hits[0].parent if bin_hits else None
print("corpus:", CORPUS)

pkg = WORK / "minivigpt"
if pkg.exists():
    shutil.rmtree(pkg)
shutil.copytree(SRC / "minivigpt", pkg)
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
print("package:", pkg)


In [ ]:
# Kaggle images ship torch/numpy/tqdm; make sure the rest is present.
import importlib.util
need = {"yaml": "PyYAML>=6.0", "datasets": "datasets>=3.0", "tokenizers": "tokenizers>=0.20"}
missing = [p for m, p in need.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("deps ok:", missing or "nothing to install")

In [ ]:
import torch, yaml
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

# This PyTorch build dropped sm_60, so a P100 would fail with
# "no kernel image is available" on the first CUDA op. Fail loudly here instead.
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability(0)
    supported = torch.cuda.get_arch_list()
    print("gpu:", name, "| capability: sm_%d%d" % (major, minor))
    print("torch supports:", supported)
    if "sm_%d%d" % (major, minor) not in supported:
        raise SystemExit(
            "GPU %s (sm_%d%d) is not supported by this PyTorch build. "
            "Set the accelerator to T4 in Settings > Accelerator and re-run."
            % (name, major, minor)
        )
    print("bf16:", torch.cuda.is_bf16_supported())
else:
    print("WARNING: no GPU - training will be very slow")

if CONFIG_SRC is None:
    raise SystemExit("config.yaml not found under /kaggle/input")
cfg = yaml.safe_load(CONFIG_SRC.read_text(encoding="utf-8"))
cfg["output_dir"] = str(WORK / "minivigpt_outputs")
if CORPUS is not None:
    cfg.setdefault("dataset", {})["prepared_dir"] = str(CORPUS)
config_path = WORK / "run_config.yaml"
config_path.write_text(yaml.safe_dump(cfg, allow_unicode=True), encoding="utf-8")
print("steps:", cfg["training"]["max_steps"], "| output:", cfg["output_dir"])
print("prepared_dir:", cfg.get("dataset", {}).get("prepared_dir"))


In [ ]:
# Resume automatically if a previous run's checkpoint was attached as a dataset.
resume = None
for q in sorted(INPUT.rglob("checkpoint_latest.pt")):
    resume = str(q)
    break
print("resume:", resume or "starting fresh")


In [ ]:
from minivigpt.train import train
output_dir = train(config_path, resume=resume)
print("done:", output_dir)

In [ ]:
import json
out = Path(cfg["output_dir"])
summary = json.loads((out / "summary.json").read_text(encoding="utf-8"))
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
from minivigpt.generate import generate_text

s = cfg["sampling"]
prompts = [s["prompt"], "Hà Nội là", "Học máy là một lĩnh vực"]
for prompt in prompts:
    text = generate_text(
        checkpoint=out / "checkpoint_best.pt",
        tokenizer_path=out / "tokenizer.json",
        prompt=prompt,
        max_new_tokens=int(s["max_new_tokens"]),
        temperature=float(s["temperature"]),
        top_k=int(s["top_k"]),
        top_p=float(s.get("top_p", 0.95)),
    )
    print("---", repr(prompt))
    print(text)
    print()
